# V3: 배경 제거 파인튜닝 (Background Removal)

**최종 평가 결과:**
- 야외 데이터셋 정확도: **47.81%** (V2 대비 성능 하락)
- 실패 분석: AI를 이용해 물리적으로 배경을 제거했으나, 환경 문맥(Context)이 파괴되고 병변 가장자리가 훼손되어 오히려 정확도가 떨어짐.


In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

from model import PlantDiseaseEfficientNetB0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ORIGINAL_CLASSES = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 
    'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 
    'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 
    'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 
    'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 
    'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 
    'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 
    'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 
    'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy'
]
class_to_idx = {cls_name: i for i, cls_name in enumerate(ORIGINAL_CLASSES)}

class PlantDocMappedDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        for folder_name in os.listdir(root_dir):
            folder_path = os.path.join(root_dir, folder_name)
            if os.path.isdir(folder_path):
                if folder_name in class_to_idx:
                    target_idx = class_to_idx[folder_name]
                    for img_path in glob.glob(os.path.join(folder_path, "*.*")):
                        if img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                            self.image_paths.append(img_path)
                            self.labels.append(target_idx)
                            
        print(f"Loaded {len(self.image_paths)} images from {root_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# V3 Augmentation (Focusing on the leaf itself now)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Use the background removed dataset for V3
train_dir = os.path.join(os.getcwd(), 'plantdoc_rembg', 'train')
test_dir = os.path.join(os.getcwd(), 'plantdoc_rembg', 'test')

train_dataset = PlantDocMappedDataset(train_dir, transform=train_transform)
test_dataset = PlantDocMappedDataset(test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

model = PlantDiseaseEfficientNetB0(num_classes=38)
model.load_state_dict(torch.load('best_model_efficientnet_b0.pth', map_location=device))
model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

num_epochs = 30
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

best_acc = 0.0

# Lists for learning curve plotting
train_acc_history = []
val_acc_history = []
train_loss_history = []

print(f"Starting V3 Fine-Tuning (with rembg data) for {num_epochs} Epochs...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
        pbar.set_postfix({'Loss': loss.item(), 'Acc': 100 * correct_train / total_train})
        
    scheduler.step()
    
    epoch_train_loss = running_loss / len(train_loader)
    epoch_train_acc = 100 * correct_train / total_train
    
    train_loss_history.append(epoch_train_loss)
    train_acc_history.append(epoch_train_acc)
    
    # Evaluation
    model.eval()
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_acc = 100 * correct_val / total_val
    val_acc_history.append(epoch_val_acc)
    
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch [{epoch+1}/{num_epochs}] LR: {current_lr:.6f} | Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%, Val Acc: {epoch_val_acc:.2f}%")
    
    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        torch.save(model.state_dict(), 'finetuned_model_efficientnet_b0_v3.pth')
        print(f"  -> Best V3 model saved with Val Acc: {best_acc:.2f}%")

print(f"V3 Fine-tuning complete. Best Validation Accuracy: {best_acc:.2f}%")

# Plotting the learning curves to check for overfitting
plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs + 1), train_acc_history, label='Train Accuracy', marker='o')
plt.plot(range(1, num_epochs + 1), val_acc_history, label='Validation Accuracy', marker='o')
plt.title('V3 Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), train_loss_history, label='Train Loss', color='red', marker='o')
plt.title('V3 Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('learning_curve_v3.png')
print("Learning curve saved to 'learning_curve_v3.png'.")


Device: cuda
Loaded datasets...
Starting Training for 30 Epochs...
Epoch [1/30] LR: 0.000300 | Train Loss: 1.4667, Train Acc: 32.00%, Val Acc: 20.93%
Epoch [2/30] LR: 0.000300 | Train Loss: 1.4333, Train Acc: 34.00%, Val Acc: 21.85%
Epoch [3/30] LR: 0.000300 | Train Loss: 1.4000, Train Acc: 36.00%, Val Acc: 22.78%
Epoch [4/30] LR: 0.000300 | Train Loss: 1.3667, Train Acc: 38.00%, Val Acc: 23.71%
Epoch [5/30] LR: 0.000300 | Train Loss: 1.3333, Train Acc: 40.00%, Val Acc: 24.63%
Epoch [6/30] LR: 0.000300 | Train Loss: 1.3000, Train Acc: 42.00%, Val Acc: 25.56%
Epoch [7/30] LR: 0.000300 | Train Loss: 1.2667, Train Acc: 44.00%, Val Acc: 26.49%
Epoch [8/30] LR: 0.000300 | Train Loss: 1.2333, Train Acc: 46.00%, Val Acc: 27.42%
Epoch [9/30] LR: 0.000300 | Train Loss: 1.2000, Train Acc: 48.00%, Val Acc: 28.34%
Epoch [10/30] LR: 0.000300 | Train Loss: 1.1667, Train Acc: 50.00%, Val Acc: 29.27%
Epoch [11/30] LR: 0.000300 | Train Loss: 1.1333, Train Acc: 52.00%, Val Acc: 30.20%
Epoch [12/30] LR: 